In [13]:
# 02_train_compare.ipynb — первая ячейка (импорты)
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
import time
import matplotlib.pyplot as plt
from tqdm import tqdm  # добавь в начало ячейки с импортами

DATA_PATH = r"../data/raw/ogyeiv2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [14]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ CUDA не найден. GPU нет или не установлен драйвер.")

PyTorch version: 2.7.1+cu118
CUDA available: True
GPU name: NVIDIA GeForce RTX 4060


In [15]:
# Вторая ячейка — Dataset (читает картинки и первый параметр из txt)
class PillDataset(Dataset):
    def __init__(self, data_path, split='train', transform=None):
        self.images_dir = f"{data_path}/{split}/images"
        self.labels_dir = f"{data_path}/{split}/labels"
        self.files = [f for f in os.listdir(self.labels_dir) if f.endswith('.txt')]
        self.transform = transform
        
        # Загружаем названия классов для вывода
        df_names = pd.read_csv(f"{data_path}/extracted_sentences.csv", header=None)
        self.id_to_name = dict(zip(range(len(df_names)), df_names[0].tolist()))
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        txt_file = self.files[idx]
        # БЕРЁМ ТОЛЬКО ПЕРВОЕ ЧИСЛО ИЗ TXT
        with open(f"{self.labels_dir}/{txt_file}", 'r') as f:
            class_id = int(float(f.read().strip().split()[0]))
        
        img_file = txt_file.replace('.txt', '.jpg')
        image = Image.open(f"{self.images_dir}/{img_file}").convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, class_id, self.id_to_name.get(class_id, f"class_{class_id}")

In [16]:
# Третья ячейка — трансформации
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Загрузка данных
train_dataset = PillDataset(DATA_PATH, 'train', train_transform)
val_dataset = PillDataset(DATA_PATH, 'valid', val_transform)
test_dataset = PillDataset(DATA_PATH, 'test', val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)} images, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
print(f"Number of classes: {len(set([train_dataset.id_to_name[k] for k in train_dataset.id_to_name]))}")

Train: 3136 images, Val: 672, Test: 672
Number of classes: 112


In [22]:
# ============================================
# 4. СОЗДАНИЕ DATALOADER (с pin_memory)
# ============================================
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train: {len(train_dataset)} images, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# ============================================
# 5. ПРОВЕРКА GPU
# ============================================
print(f"Device: {DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ============================================
# 6. ФУНКЦИЯ ОБУЧЕНИЯ (с правильной передачей на GPU)
# ============================================
def train_simple(model, model_name, num_epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    
    for epoch in range(num_epochs):
        print(f"\n--- {model_name} Epoch {epoch+1}/{num_epochs} ---")
        
        # ОБУЧЕНИЕ
        model.train()
        train_loss = 0
        for i, (images, labels, _) in enumerate(train_loader):
            # ПРИНУДИТЕЛЬНО ПЕРЕНОСИМ НА GPU
            images = images.cuda(non_blocking=True)
            labels = labels.cuda(non_blocking=True)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
            if i % 20 == 0:
                print(f"  Batch {i}, Loss: {loss.item():.4f}")
        
        avg_train_loss = train_loss / len(train_loader)
        print(f"  Средний Train Loss: {avg_train_loss:.4f}")
        
        # ВАЛИДАЦИЯ
        model.eval()
        correct = 0
        total = 0
        val_loss = 0
        with torch.no_grad():
            for images, labels, _ in val_loader:
                images = images.cuda(non_blocking=True)
                labels = labels.cuda(non_blocking=True)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_acc = correct / total
        print(f"  Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    return val_acc

# ============================================
# 7. СОЗДАНИЕ МОДЕЛЕЙ
# ============================================
print("\nСоздание моделей...")
models_dict = {
    'ResNet50': models.resnet50(pretrained=True),
    'EfficientNet_B0': models.efficientnet_b0(pretrained=True),
    'MobileNet_V3': models.mobilenet_v3_large(pretrained=True)
}

# Замена последних слоёв
models_dict['ResNet50'].fc = nn.Linear(2048, 112)
models_dict['EfficientNet_B0'].classifier[1] = nn.Linear(1280, 112)
models_dict['MobileNet_V3'].classifier[3] = nn.Linear(1280, 112)

# Перенос на GPU
for name in models_dict:
    models_dict[name] = models_dict[name].cuda()
print("✅ Модели созданы и перенесены на GPU")

# ============================================
# 8. ОБУЧЕНИЕ
# ============================================
results = {}
for name, model in models_dict.items():
    print(f"\n{'='*50}")
    print(f"Обучение {name}")
    print(f"{'='*50}")
    acc = train_simple(model, name, num_epochs=5)
    results[name] = acc

# ============================================
# 9. РЕЗУЛЬТАТЫ
# ============================================
print("\n" + "="*50)
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
print("="*50)
for name, acc in results.items():
    print(f"{name}: {acc:.4f}")

Train: 3136 images, Val: 672, Test: 672
Device: cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 4060

Создание моделей...
✅ Модели созданы и перенесены на GPU

Обучение ResNet50

--- ResNet50 Epoch 1/5 ---
  Batch 0, Loss: 4.7401
  Batch 20, Loss: 4.2431
  Batch 40, Loss: 3.8077
  Batch 60, Loss: 3.1980
  Batch 80, Loss: 2.4844
  Средний Train Loss: 3.5027
  Val Loss: 2.0831, Val Acc: 0.6057

--- ResNet50 Epoch 2/5 ---
  Batch 0, Loss: 1.7910


KeyboardInterrupt: 

In [18]:
# Пятая ячейка — создание и обучение трёх моделей
models_dict = {
    'ResNet50': models.resnet50(pretrained=True),
    'EfficientNet_B0': models.efficientnet_b0(pretrained=True),
    'MobileNet_V3': models.mobilenet_v3_large(pretrained=True)
}

# Замена последнего слоя под 112 классов
num_classes = 112
models_dict['ResNet50'].fc = nn.Linear(2048, num_classes)
models_dict['EfficientNet_B0'].classifier[1] = nn.Linear(1280, num_classes)
models_dict['MobileNet_V3'].classifier[3] = nn.Linear(1280, num_classes)

# Перенос на GPU
for name, model in models_dict.items():
    models_dict[name] = model.to(DEVICE)

# Обучение
results = {}
for name, model in models_dict.items():
    print(f"\n{'='*50}\nTraining {name}\n{'='*50}")
    history, best_acc = train_model(model, name, num_epochs=15)
    results[name] = {'history': history, 'best_val_acc': best_acc}


Training ResNet50


Training ResNet50:   0%|          | 0/15 [00:59<?, ?it/s]


KeyboardInterrupt: 